
# Session 4: Practical Machine Learning Issues  
## Beginner-friendly tips for real-world machine learning

**Course theme:** Practical machine learning problems that appear in real projects  
**Suggested duration:** 6 hours  
**Level:** Beginner to lower-intermediate  
**Main library:** scikit-learn  

---

## Why this module matters

In practice, machine learning is usually **not** just:

- choose a model
- call `.fit()`
- report one score

Real projects often involve issues like:

- messy features
- categorical variables
- missing values
- scaling problems
- imbalanced classes
- overfitting
- hyperparameter tuning
- model comparison

This notebook focuses on those practical issues in a beginner-friendly way.

---

## Main topics covered

- feature engineering
- dummy variables / one-hot encoding
- handling missing values
- data standardization
- train/test split and data leakage
- class imbalance
- class weights
- hyperparameter tuning
- grid search
- pipelines
- model comparison
- practical workflow habits



## Learning objectives

By the end of this module, students should be able to:

- explain why practical preprocessing matters in machine learning
- create simple new features from existing columns
- handle categorical variables with dummy variables / one-hot encoding
- explain why the dummy variable trap can matter in some models
- standardize numeric features when appropriate
- identify and handle missing values
- recognize class imbalance and explain why accuracy can be misleading
- use class weights in a classification problem
- tune hyperparameters with GridSearchCV
- build clean workflows with pipelines
- compare multiple models in a practical and structured way



## Suggested 6-hour teaching plan

### Hour 1
- Practical ML workflow overview
- Load messy dataset
- Missing values
- Numeric vs categorical features

### Hour 2
- Feature engineering
- Dummy variables / one-hot encoding
- Why some models need careful preprocessing

### Hour 3
- Standardization
- Data leakage
- Pipelines

### Hour 4
- Imbalanced data
- Accuracy vs precision/recall
- Class weights

### Hour 5
- Hyperparameters
- Grid search
- Model comparison

### Hour 6
- End-to-end pipeline
- Practical checklist
- Exercises and reflection


In [ ]:

# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report
)

%matplotlib inline



# Part 1: Load a practical dataset

We will use a customer-risk classification dataset built for teaching.

It includes:

- numeric features
- categorical features
- missing values
- class imbalance

**File used:** `practical_ml_customer_risk.csv`

Target column:
- `high_risk`


In [ ]:

# Load dataset
df = pd.read_csv("/mnt/data/practical_ml_customer_risk.csv")

# Show first rows
df.head()


In [ ]:

# Shape and data types
print("Rows and columns:", df.shape)
print()
print(df.dtypes)


In [ ]:

# Missing values by column
df.isna().sum()


In [ ]:

# Class balance
df["high_risk"].value_counts()


In [ ]:

# Plot class balance
class_counts = df["high_risk"].value_counts().sort_index()

plt.figure(figsize=(5, 4))
plt.bar(class_counts.index.astype(str), class_counts.values)
plt.xlabel("High Risk")
plt.ylabel("Count")
plt.title("Class Distribution")
plt.show()



## Practical lesson 1: start by inspecting the data

Before choosing a model, ask:

- Which columns are numeric?
- Which columns are categorical?
- Are there missing values?
- Is the target imbalanced?
- Are any obvious new features useful?

This is often more important than jumping to a fancy model too early.



# Part 2: Feature engineering

**Feature engineering** means creating new useful input variables from existing ones.

Good feature engineering often helps more than switching from one model to another.

Examples:
- spending per product
- recent inactivity flag
- customer age group
- average support burden

The main rule for beginners:

> Create features that make business sense.


In [ ]:

# Create a copy for manual feature engineering demos
df_fe = df.copy()

# Create a few simple engineered features
df_fe["spend_per_product"] = df_fe["monthly_spend"] / df_fe["num_products"]
df_fe["is_inactive"] = (df_fe["days_since_login"] > 20).astype(int)
df_fe["tickets_per_product"] = df_fe["support_tickets"] / df_fe["num_products"]

df_fe[[
    "monthly_spend", "num_products", "spend_per_product",
    "days_since_login", "is_inactive",
    "support_tickets", "tickets_per_product"
]].head()



## Good beginner questions for feature engineering

- Does this new feature summarize something meaningful?
- Does it reflect behavior the business cares about?
- Could it help a model detect a useful pattern?
- Would the feature also be available in the future when making predictions?

That last question is very important.

If a feature would not be available at prediction time, using it can create a real deployment problem.



# Part 3: Dummy variables and one-hot encoding

Many machine learning models cannot directly use text categories like:

- `"Monthly"`
- `"Annual"`
- `"West"`
- `"Mobile"`

So we convert them into numeric indicator columns.

Example:

`contract_type`

becomes something like:

- `contract_type_Monthly`
- `contract_type_Annual`
- `contract_type_Two-Year`

This process is called:

- dummy variables
- one-hot encoding


In [ ]:

# Small demo with pandas.get_dummies()
demo_df = pd.DataFrame({
    "contract_type": ["Monthly", "Annual", "Two-Year", "Monthly"]
})

pd.get_dummies(demo_df, dtype=int)



## Why can dummy variables matter?

Suppose we have 3 categories:

- Monthly
- Annual
- Two-Year

If we create all 3 dummy columns and also include an intercept in a linear-type model, then one column can be perfectly predicted from the others.

This is often called the **dummy variable trap**.

In practice:
- many libraries can still handle it
- but it is good to understand the idea
- sometimes we intentionally drop one category as the reference group


In [ ]:

# Demo: dropping one category
pd.get_dummies(demo_df, drop_first=True, dtype=int)



In scikit-learn, `OneHotEncoder` is often preferred inside pipelines because it keeps preprocessing cleaner and more reusable.



# Part 4: Missing values

Missing values are common in real projects.

Beginner-friendly strategies:

### Numeric columns
- fill with mean
- fill with median

### Categorical columns
- fill with most frequent category
- or create an `"Unknown"` category

Important:

> Always learn imputation values from the training set only, not from the full dataset.

Otherwise, you risk **data leakage**.



# Part 5: Data leakage

**Data leakage** means information from outside the training process accidentally helps the model.

Examples:
- scaling using the full dataset before splitting
- imputing with values computed from the full dataset
- using future information that would not be available in real life

Leakage makes the model look better than it really is.

This is one of the most common beginner mistakes.


In [ ]:

# Define features and target
X = df_fe.drop(columns="high_risk")
y = df_fe["high_risk"]

# Split first, before fitting preprocessing steps
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training set:", X_train.shape)
print("Test set:", X_test.shape)



# Part 6: Standardization

Some models care a lot about feature scale.

Examples:
- Logistic Regression
- KNN
- SVM

If one feature is much larger than another, it can dominate the model.

Standardization usually transforms a numeric feature as:

\[
x_{scaled} = \frac{x - \text{mean}}{\text{std}}
\]

This makes features easier to compare on a similar scale.

Tree-based models often do **not** need standardization in the same way.


In [ ]:

# Separate numeric and categorical features
numeric_features = X_train.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X_train.select_dtypes(include=["object"]).columns.tolist()

print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)


In [ ]:

# Preprocessing for numeric columns
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),   # fill missing numeric values
    ("scaler", StandardScaler())                     # standardize numeric values
])

# Preprocessing for categorical columns
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),   # fill missing categories
    ("onehot", OneHotEncoder(handle_unknown="ignore"))      # convert categories to dummy variables
])

# Combine preprocessing
preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

preprocessor



## Why pipelines are so useful

A pipeline helps us keep steps in the correct order.

Instead of manually remembering:

1. impute
2. encode
3. scale
4. fit model

we let the pipeline manage the workflow.

Pipelines help with:
- cleaner code
- fewer mistakes
- less data leakage
- easier grid search



# Part 7: Build a clean baseline model

We will start with a practical baseline:

- preprocessing
- Logistic Regression


In [ ]:

# Baseline pipeline with Logistic Regression
baseline_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=1000))
])

baseline_model.fit(X_train, y_train)

print("Baseline model trained.")


In [ ]:

# Predictions
baseline_pred = baseline_model.predict(X_test)
baseline_prob = baseline_model.predict_proba(X_test)[:, 1]

# Basic metrics
print("Accuracy :", round(accuracy_score(y_test, baseline_pred), 4))
print("Precision:", round(precision_score(y_test, baseline_pred), 4))
print("Recall   :", round(recall_score(y_test, baseline_pred), 4))
print("F1 Score :", round(f1_score(y_test, baseline_pred), 4))
print("ROC AUC  :", round(roc_auc_score(y_test, baseline_prob), 4))


In [ ]:

# Confusion matrix
cm = confusion_matrix(y_test, baseline_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot()
plt.title("Confusion Matrix - Baseline Logistic Regression")
plt.show()



# Part 8: Imbalanced data

In real classification problems, one class is often rarer than the other.

Examples:
- fraud
- disease
- churn
- default
- defects

This is called **class imbalance**.

---

## Why imbalance is a problem

If the positive class is rare, a model can get high accuracy by mostly predicting the majority class.

So for imbalanced data, we often pay more attention to:
- precision
- recall
- F1
- ROC AUC
- confusion matrix


In [ ]:

# Show class proportions
y.value_counts(normalize=True)



## One practical option: class weights

Some models, such as Logistic Regression and Decision Trees, support `class_weight`.

This tells the model to care more about mistakes on the minority class.

A common beginner-friendly option is:

- `class_weight="balanced"`


In [ ]:

# Logistic Regression with class_weight='balanced'
balanced_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=1000, class_weight="balanced"))
])

balanced_model.fit(X_train, y_train)

balanced_pred = balanced_model.predict(X_test)
balanced_prob = balanced_model.predict_proba(X_test)[:, 1]

print("Accuracy :", round(accuracy_score(y_test, balanced_pred), 4))
print("Precision:", round(precision_score(y_test, balanced_pred), 4))
print("Recall   :", round(recall_score(y_test, balanced_pred), 4))
print("F1 Score :", round(f1_score(y_test, balanced_pred), 4))
print("ROC AUC  :", round(roc_auc_score(y_test, balanced_prob), 4))



Students should notice:

- precision may go down
- recall may go up
- the “best” choice depends on business cost

For example:
- in fraud detection, missing fraud may be worse than extra false alarms
- in medical screening, missing a disease may be worse than over-testing



# Part 9: Hyperparameters

A **parameter** is learned by the model from data.

Examples:
- coefficients in Logistic Regression
- split rules in a tree

A **hyperparameter** is chosen by us before training.

Examples:
- regularization strength in Logistic Regression
- tree depth in a Decision Tree
- number of neighbors in KNN

Hyperparameters can strongly affect performance.



## Grid search idea

Grid search tries many combinations of hyperparameters and checks which one performs best.

Instead of guessing one value, we define a small search grid.

For example:
- `C = [0.1, 1, 10]`
- `class_weight = [None, "balanced"]`

Then GridSearchCV tests them using cross-validation.

This is a practical and common workflow.



# Part 10: Hyperparameter tuning with GridSearchCV

We will tune a Logistic Regression model.

Important point for beginners:

- preprocessing stays inside the pipeline
- grid search works on the whole pipeline
- this keeps the workflow safe and clean


In [ ]:

# Pipeline for tuning
tune_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=1000))
])

# Parameter grid
param_grid = {
    "model__C": [0.1, 1, 5, 10],                  # inverse regularization strength
    "model__class_weight": [None, "balanced"]     # handle imbalance or not
}

grid = GridSearchCV(
    tune_model,
    param_grid=param_grid,
    cv=5,
    scoring="f1"
)

grid.fit(X_train, y_train)

print("Best parameters:", grid.best_params_)
print("Best CV score:", round(grid.best_score_, 4))


In [ ]:

# Evaluate the best tuned model
best_model = grid.best_estimator_

best_pred = best_model.predict(X_test)
best_prob = best_model.predict_proba(X_test)[:, 1]

print("Accuracy :", round(accuracy_score(y_test, best_pred), 4))
print("Precision:", round(precision_score(y_test, best_pred), 4))
print("Recall   :", round(recall_score(y_test, best_pred), 4))
print("F1 Score :", round(f1_score(y_test, best_pred), 4))
print("ROC AUC  :", round(roc_auc_score(y_test, best_prob), 4))



## What does `C` mean in Logistic Regression?

For scikit-learn Logistic Regression:

- small `C` → stronger regularization
- large `C` → weaker regularization

So `C` controls how flexible the model becomes.

This is a practical tuning point students often see in real work.



# Part 11: Compare baseline vs balanced vs tuned models


In [ ]:

comparison = pd.DataFrame([
    {
        "model": "Baseline Logistic",
        "accuracy": accuracy_score(y_test, baseline_pred),
        "precision": precision_score(y_test, baseline_pred),
        "recall": recall_score(y_test, baseline_pred),
        "f1": f1_score(y_test, baseline_pred),
        "roc_auc": roc_auc_score(y_test, baseline_prob)
    },
    {
        "model": "Balanced Logistic",
        "accuracy": accuracy_score(y_test, balanced_pred),
        "precision": precision_score(y_test, balanced_pred),
        "recall": recall_score(y_test, balanced_pred),
        "f1": f1_score(y_test, balanced_pred),
        "roc_auc": roc_auc_score(y_test, balanced_prob)
    },
    {
        "model": "Tuned Logistic",
        "accuracy": accuracy_score(y_test, best_pred),
        "precision": precision_score(y_test, best_pred),
        "recall": recall_score(y_test, best_pred),
        "f1": f1_score(y_test, best_pred),
        "roc_auc": roc_auc_score(y_test, best_prob)
    }
])

comparison.sort_values("f1", ascending=False)



This is a very realistic lesson:

> different models or settings can be better depending on the metric that matters.

If the business wants:
- fewer missed positives → focus more on recall
- fewer false alarms → focus more on precision
- balanced performance → focus on F1



# Part 12: A Decision Tree practical example

Decision Trees handle mixed features differently and do not need standardization in the same way.

But they still benefit from:
- clean preprocessing
- missing value handling
- tuning


In [ ]:

# Simpler preprocessing for tree model
tree_numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

tree_categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

tree_preprocessor = ColumnTransformer(transformers=[
    ("num", tree_numeric_transformer, numeric_features),
    ("cat", tree_categorical_transformer, categorical_features)
])

tree_pipeline = Pipeline(steps=[
    ("preprocessor", tree_preprocessor),
    ("model", DecisionTreeClassifier(random_state=42))
])

tree_param_grid = {
    "model__max_depth": [3, 5, 7, None],
    "model__min_samples_split": [2, 10, 20],
    "model__class_weight": [None, "balanced"]
}

tree_grid = GridSearchCV(
    tree_pipeline,
    param_grid=tree_param_grid,
    cv=5,
    scoring="f1"
)

tree_grid.fit(X_train, y_train)

print("Best tree parameters:", tree_grid.best_params_)
print("Best tree CV score:", round(tree_grid.best_score_, 4))


In [ ]:

# Evaluate tuned tree
best_tree = tree_grid.best_estimator_
tree_pred = best_tree.predict(X_test)
tree_prob = best_tree.predict_proba(X_test)[:, 1]

print("Accuracy :", round(accuracy_score(y_test, tree_pred), 4))
print("Precision:", round(precision_score(y_test, tree_pred), 4))
print("Recall   :", round(recall_score(y_test, tree_pred), 4))
print("F1 Score :", round(f1_score(y_test, tree_pred), 4))
print("ROC AUC  :", round(roc_auc_score(y_test, tree_prob), 4))



# Part 13: Practical checklist for beginners

Before training:
- understand the target
- inspect missing values
- separate numeric and categorical columns
- think about useful engineered features
- split into train/test early

During preprocessing:
- impute missing values
- encode categorical features
- standardize numeric data when needed
- avoid data leakage

During modeling:
- start with a baseline
- choose metrics that match the business goal
- check class balance
- try class weights when needed
- tune important hyperparameters

After modeling:
- compare multiple models
- interpret results carefully
- do not trust accuracy alone
- keep the workflow reproducible



# Part 14: Common beginner mistakes

1. **Using the full dataset before splitting**  
   This can create leakage.

2. **Forgetting categorical encoding**  
   Many models cannot use raw text categories.

3. **Ignoring missing values**  
   Some models will fail, others will behave poorly.

4. **Scaling tree models unnecessarily and forgetting to scale distance-based models**  
   Different models have different needs.

5. **Looking only at accuracy**  
   Especially risky for imbalanced data.

6. **Tuning hyperparameters without a clear metric goal**  
   Always ask: what are we optimizing for?

7. **Creating features that would not exist at prediction time**  
   This creates deployment problems.



# Part 15: Hands-on exercises

## Exercise 1
Create one new feature that you think may help prediction.

## Exercise 2
Try `drop='first'` in one-hot encoding.  
What changes conceptually?

## Exercise 3
Run GridSearchCV again, but optimize for:
- `precision`
- `recall`

How do the best parameters change?

## Exercise 4
Remove the engineered features and compare results.

## Exercise 5
Train a Logistic Regression model without `class_weight="balanced"`.  
Compare recall and precision.

## Exercise 6
Explain in plain English:
- feature engineering
- one-hot encoding
- standardization
- class imbalance
- hyperparameter tuning

## Exercise 7
Which practical issue do you think is most dangerous in real life:
- leakage
- imbalance
- missing values
- poor feature engineering
Why?


In [ ]:

# Starter code for Exercise 3
for metric_name in ["precision", "recall"]:
    grid_temp = GridSearchCV(
        tune_model,
        param_grid=param_grid,
        cv=5,
        scoring=metric_name
    )
    grid_temp.fit(X_train, y_train)
    print("Scoring metric:", metric_name)
    print("Best parameters:", grid_temp.best_params_)
    print("Best CV score:", round(grid_temp.best_score_, 4))
    print()



# Part 16: Wrap-up

## Key takeaways

- Practical machine learning is mostly about making good workflow decisions
- Feature engineering can be very powerful
- Dummy variables / one-hot encoding are essential for categorical features
- Standardization matters for some models
- Pipelines help prevent mistakes and leakage
- Imbalanced data requires careful metric choice
- Hyperparameter tuning helps improve models systematically
- Grid search is a practical and common beginner-friendly tuning method
- In real life, good preprocessing and evaluation often matter more than chasing fancy models



## End of notebook

This module is designed as a practical “survival guide” for beginner machine learning work.

A natural next step after this module could be:
- ensemble models
- random forests and gradient boosting
- model interpretation tools
- deployment and monitoring
